The closing market value for a Monday is processed and published to FRED on Tuesday morning around 6:37 AM PST. Data for Friday becomes available on Monday morning.

In [1]:
from datetime import datetime
import pandas as pd
from fredapi import Fred

# Load API key from .env file
env_path = r"C:\Users\ping\Files_win10\python\py311\.env\my_api_key.env"
api_key = None

with open(env_path, "r") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")  # Remove surrounding quotes
            if key == "Fred":
                api_key = value
                break

if not api_key:
    raise ValueError("API key 'Fred' not found in .env file")

fred = Fred(api_key=api_key)

# FRED Tickers: VXVCLS = VIX3M, VIXCLS = VIX
tickers_map = {"VXVCLS": "^VIX3M", "VIXCLS": "^VIX"}
start_date = datetime(1970, 1, 1)
end_date = datetime.now()

# Fetch each series
series_dict = {}
for fred_ticker, target_symbol in tickers_map.items():
    series_dict[target_symbol] = fred.get_series(
        fred_ticker, observation_start=start_date, observation_end=end_date
    )

# Create wide DataFrame with DatetimeIndex
df_wide = pd.DataFrame(series_dict)
df_wide.index.name = "Date"

# Reshape wide data to long MultiIndex (Ticker, Date) format
df_long = df_wide.unstack().reset_index()
df_long.columns = ["Ticker", "Date", "Adj Close"]

# Drop non-valid/NaN observation rows
df_long = df_long.dropna(subset=["Adj Close"]).copy()

# Populate remaining OHLCV float64 columns matching target format
df_long["Adj Open"] = df_long["Adj Close"].astype(float)
df_long["Adj High"] = df_long["Adj Close"].astype(float)
df_long["Adj Low"] = df_long["Adj Close"].astype(float)
df_long["Adj Close"] = df_long["Adj Close"].astype(float)
df_long["Volume"] = 0.0

# Reorder columns precisely to target format
cols = ["Adj Open", "Adj High", "Adj Low", "Adj Close", "Volume"]
df_final = df_long.set_index(["Ticker", "Date"])[cols].sort_index()

print("Data successfully downloaded and formatted.")
print("\nDataFrame Info:")
print(df_final.info())

# Save to CSV and Parquet to path_parent_data_dir
path_parent_data_dir = r"C:\Users\ping\Files_win10\python\py311\stocks\data\VIX3M_VIX"
df_final.to_csv(path_parent_data_dir + ".csv")
df_final.to_parquet(path_parent_data_dir + ".parquet", engine="pyarrow")
print(f"\nSaved to '{path_parent_data_dir}.csv'")
print(f"Saved to '{path_parent_data_dir}.parquet'")

df_final

Data successfully downloaded and formatted.

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
MultiIndex: 14026 entries, ('^VIX', Timestamp('1990-01-02 00:00:00')) to ('^VIX3M', Timestamp('2026-10-05 00:00:00'))
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Open   14026 non-null  float64
 1   Adj High   14026 non-null  float64
 2   Adj Low    14026 non-null  float64
 3   Adj Close  14026 non-null  float64
 4   Volume     14026 non-null  float64
dtypes: float64(5)
memory usage: 919.8+ KB
None

Saved to 'C:\Users\ping\Files_win10\python\py311\stocks\data\VIX3M_VIX.csv'
Saved to 'C:\Users\ping\Files_win10\python\py311\stocks\data\VIX3M_VIX.parquet'


Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^VIX   1990-01-02     17.24     17.24    17.24      17.24     0.0
       1990-01-03     18.19     18.19    18.19      18.19     0.0
       1990-01-04     19.22     19.22    19.22      19.22     0.0
       1990-01-05     20.11     20.11    20.11      20.11     0.0
       1990-01-08     20.26     20.26    20.26      20.26     0.0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-09-29     18.09     18.09    18.09      18.09     0.0
       2026-09-30     18.37     18.37    18.37      18.37     0.0
       2026-10-01     18.58     18.58    18.58      18.58     0.0
       2026-10-02     18.01     18.01    18.01      18.01     0.0
       2026-10-05     18.00     18.00    18.00      18.00     0.0

[14026 rows x 5 columns]